# Part 11 · Notebook 07 — Event analysis: post-earnings drift and an early-warning overlay

**Sessions:** S7 (Event & crash analysis) · [Lesson plan](../../docs/lessons/PART_11_AI_NLP_COWORK.md) · graded labs in [`labs/part11/`](../../labs/part11/)

**You will:**
1. Measure the drift after earnings by the guidance label an LLM extracted.
2. Trade it with costs, and see how fast costs eat it.
3. Build a composite stress score from each feature's own past, with no look-ahead.
4. Use it only to cut risk, and measure the drawdown it saves.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Nothing calls the API: model outputs are recorded responses built as real `anthropic` Message objects, and the data is synthetic with known answers, so every check is exact and free.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p11lib.py is in notebooks/part11/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p11lib as p

p.use_course_style()

## 1. Drift by guidance label

Notebook 01 extracted each call's guidance change (raised, maintained, lowered). `p.earnings_market()` plants a known post-earnings drift: after "raised" the stock drifts up for 5 days from the day **after** the call, after "lowered" down, after "maintained" nothing. `p.drift_by_label` pools market-model event studies per label:

In [ ]:
R, mkt, events = p.earnings_market()
p.drift_by_label(R, mkt, events).round(4)

## 2. Trade it

Long after "raised", short after "lowered", entering at the close of the event day and holding `hold` days (the returns of days `d+1 … d+hold`). Net return per trade = `side × Σ returns − cost_bps/1e4` (one round trip).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def pead_trades(R, events, hold=5, cost_bps=10.0):
    out = {}
    for i, e in events.iterrows():
        side = {"raised": 1, "lowered": -1}.get(e["label"])
        if side is None:
            continue                                                 # "maintained": no trade
        d = R.index.get_loc(e["date"])
        out[i] = ...                                                 # ✍️
    return pd.Series(out, dtype=float)

cases = [(R, events), (R, events, 3, 30.0), (R, events[events["label"] == "maintained"])]
mine = [p.attempt(pead_trades, *c) for c in cases]
mine = p.check("pead_trades", mine, [p.pead_trades(*c) for c in cases])
rows = {}
for cost in (0, 10, 30, 60):
    t = p.pead_trades(R, events, cost_bps=cost)
    rows[f"{cost} bp"] = {"trades": len(t), "mean net return": t.mean(), "hit rate": (t > 0).mean(), "t-stat": t.mean() / (t.std(ddof=1) / np.sqrt(len(t)))}
pd.DataFrame(rows).T.round(4)

The drift is strong here by construction; every 10 bp of round-trip cost takes 0.1% off each trade. Real drift is smaller and the label comes from an LLM that is sometimes wrong (notebook 01), so costs and extraction errors decide whether this survives.

## 3. An early-warning score

Stress indicators (a VIX proxy, credit spreads) are on different scales and drift over years. Score each against its **own past**: the z-score against the rolling mean and std of the `window` days ending **yesterday** (`rolling(window).mean().shift(1)`), then average across features. Using today's value in its own mean would be a small look-ahead.

In [ ]:
rng = np.random.default_rng(0)
n = 2000
stress = np.zeros(n)
for t in range(1, n):
    stress[t] = 0.98 * stress[t - 1] + rng.normal(0, 0.2)       # a persistent, hidden stress level
idx = pd.bdate_range("2015-01-01", periods=n)
vol = 0.008 * np.exp(0.6 * np.r_[0, stress[:-1]])              # volatility rises with yesterday's stress
ret = pd.Series(0.0003 + vol * rng.standard_normal(n), index=idx)
feats = pd.DataFrame({"vix_proxy": stress + rng.normal(0, 0.3, n), "credit": stress + rng.normal(0, 0.5, n)}, index=idx)

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def early_warning(features, window=250):
    m = ...                                                          # ✍️ rolling mean of the window ending yesterday
    s = ...                                                          # ✍️ rolling std of the window ending yesterday
    return ((features - m) / s).mean(axis=1)

cases = [(feats,), (feats, 100), (feats.iloc[:1000],)]
mine = [p.attempt(early_warning, *c) for c in cases]
mine = p.check("early_warning", mine, [p.early_warning(*c) for c in cases])
score = mine[0]
print(f"no look-ahead: score on the first 1,000 days computed alone equals the full-sample score: {np.allclose(mine[2], score.iloc[:1000], equal_nan=True)}")

## 4. Cut risk, don't bet

`p.overlay` halves the exposure on days after the score exceeds 1.5, and keeps it at 1 otherwise. It never goes short:

In [ ]:
over = p.overlay(ret, score)
live = slice(251, None)
dd = lambda r: float((r.cumsum() - r.cumsum().cummax()).min())             # noqa: E731
warned = score.shift(1) > 1.5
print(f"warnings on {warned.mean():.0%} of days; volatility on those days is {ret[warned].std() / ret[~warned].std():.1f}× the rest")
pd.DataFrame({"always invested": {"max drawdown": dd(ret.iloc[live]), "Sharpe": p.sharpe(ret.iloc[live])},
              "half exposure after a warning": {"max drawdown": dd(over.iloc[live]), "Sharpe": p.sharpe(over.iloc[live])}}).round(3)

The score finds the high-volatility days, and halving exposure there shrinks the drawdown more than the return: a better risk-adjusted book, without ever predicting direction. The same score as a short signal would lose, because the average return on stressed days is still positive here.

## Wrap-up

* LLM-extracted labels become event studies: measure the drift before trading it, and after costs.
* Scores against each feature's own past, with a one-day lag, are comparable and look-ahead free.
* Early warnings cut risk; they are not trade signals.
* Graded version: `labs/part11/week40_agents_ops` (`drift_by_label`, `pead_trades`, `early_warning`, `overlay`).